# Robotics Control and Learning-Guided Motion Planning

This notebook is the primary technical reference for a two-part 4-DOF robotics project: dynamic simulation and PID control in MATLAB, followed by learning-guided trajectory sampling and MPC-style control in Python. It preserves the engineering content and supplied simulation evidence while distinguishing research theory from the implementation that is present in the repository.

The intended reading order is **theory -> source code -> visual result -> interpretation**.

## 1. Evidence and provenance model

Technical claims in this notebook use the following evidence classes.

| Label | Meaning |
|---|---|
| **Verified in source** | Directly visible in the tracked MATLAB or Python implementation |
| **Reported theory** | Derivation or method described in the supplied engineering material or referenced paper |
| **Preserved result** | Figure or numerical value supplied with the project, not recomputed here |
| **Reconstructed diagnostic** | A new read-only calculation made from tracked parameters and explicitly labeled as diagnostic |
| **Supplementary explanation** | Standard engineering context added to connect equations and code |
| **Unresolved** | A fact that cannot be verified from the available source and artifacts |

This distinction is important because the preserved figures appear to have been generated from an earlier source revision in several places.

## 2. System overview

The repository contains two related but independent studies.

![Verified system architecture](../assets/images/system-architecture.png)

The Python code is an MPC-style sampled controller. It does not integrate the MATLAB dynamic model.

## 3. Robot geometry and generalized coordinates

The reported mechanism has three rotary joints and one prismatic joint, represented by

$$
q = [\theta_1,\theta_2,\theta_3,d_4]^T.
$$

![Reported kinematic structure](../assets/images/robot-arm-kinematic-structure.jpg)

The figure is a preserved project schematic. The MATLAB and Python implementations use different frame conventions, so the image should be treated as conceptual rather than as an exact frame-by-frame specification of both programs.

### 3.1 Parameter cross-check

| Quantity | MATLAB source | Python source | Interpretation |
|---|---:|---:|---|
| Link lengths | 187.5, 200, 50, 200, 112.5 | 0.1125, 0.2, 0.05, 0.2, 0.1875 | MATLAB uses millimeter-scale numbers; Python uses meter-scale numbers, with a different ordering |
| Masses | 350, 341, 100, 400, 600 | 0.6, 0.4, 0.1, 0.341, 0.35 | Stored in both parts but unused by the Python planner |
| Rotary limits | Not enforced | $[-\pi,\pi]$, then $[-\pi/2,\pi/2]$ | Verified in Python |
| Prismatic limit | Desired value 40 | 0 to 0.04 | Millimeters in MATLAB intent, meters in Python |
| Gravity | 9.81 | 9.81 | Stored in Python but not used by planning |

**Unresolved:** the historical table labels mass moments of inertia with units of $\mathrm{mm}^4$. A mass moment of inertia requires mass-length-squared units, such as $\mathrm{kg\,m^2}$ or $\mathrm{g\,mm^2}$. The original numerical values are preserved, but their physical units cannot be verified.

## 4. Part I - Lagrangian dynamics

### 4.1 Theory

The reported derivation begins with the Lagrangian

$$L(q,\dot q)=T(q,\dot q)-U(q),$$

where a rigid-body kinetic-energy contribution has the form

$$T_i=\frac{1}{2}m_i v_i^2+\frac{1}{2}I_i\dot\theta_i^2,$$

and the gravitational potential energy is

$$U=\sum_{i=1}^{5}m_i g h_i(q).$$

Euler-Lagrange equations then give

$$\frac{d}{dt}\left(\frac{\partial L}{\partial \dot q_i}\right)-\frac{\partial L}{\partial q_i}=\tau_i.$$

A conventional manipulator form is

$$M(q)\ddot q+C(q,\dot q)\dot q+G(q)=\tau.$$

**Reported-theory limitation:** the supplied derivation reduces $M$ to a constant diagonal matrix and contains an inconsistency between $m_5$ and $I_5$ in the prismatic kinetic term. It should therefore be presented as the project's simplified model, not as a complete rigid-body derivation.

### 4.2 Open-loop source implementation

The complete implementation is in [`src/matlab/part1_not_pid.m`](../src/matlab/part1_not_pid.m). The central update is: 

```matlab
inertiaMat = [inertia2 0 0 0;
              0 inertia3 0 0;
              0 0 inertia4 0;
              0 0 0 inertia5];

inputTorques = [2 2 2 5];
angularAccel = inertiaMat \ inputTorques' ...
             - inertiaMat \ (coriolisMat * anglesVel);

angle1Vel = angularAccel(1) * timeStep + angle1Vel;
angle1 = angularAccel(1) * timeStep^2 * 0.5 ...
       + angle1Vel * timeStep + angle1;
```

The variable named `coriolisMat` is a 4-by-1 expression built from gravity and configuration terms, while `anglesVel` is initialized once and never refreshed. Consequently, the source does not implement the conventional $C(q,\dot q)\dot q+G(q)$ decomposition shown above.

### 4.3 Preserved open-loop results

![Open-loop joint positions](../assets/images/open-loop-position-response.png)

![Open-loop joint velocities](../assets/images/open-loop-velocity-response.png)

![Open-loop arm configuration](../assets/images/open-loop-arm-configuration.png)

### 4.4 Open-loop interpretation

The preserved curves show monotonically increasing coordinates and velocities under constant numerical inputs. This is qualitatively consistent with a continuously accelerated, undamped numerical model. However, the figures are evidence of a historical script run rather than a validated physical response: the parameter units are inconsistent, the gravity/velocity term does not match the conventional manipulator equation, and no raw arrays or execution log were supplied.

## 5. Part I - PID control

### 5.1 Theory

For reference $r(t)$ and measured output $y(t)$,

$$e(t)=r(t)-y(t),$$

and the continuous-time PID law is

$$u(t)=K_p e(t)+K_i\int_0^t e(\tau)d\tau+K_d\frac{de(t)}{dt}.$$

Its ideal transfer function is

$$G_c(s)=K_p+\frac{K_i}{s}+K_d s.$$

The reported manual tuning values are $K_p=1.5$ for every coordinate, $K_i=[30.595,14.2,19.68,29.85]$, and $K_d=[0.005,0.005,0.005,0.01]$.

### 5.2 PID source implementation

The complete implementation is in [`src/matlab/part1_pid.m`](../src/matlab/part1_pid.m).

```matlab
K = [1.5 1.5 1.5 1.5 ...
     30.595 14.2 19.68 29.85 ...
     0.005 0.005 0.005 0.01];

errorAngles = angles_d - angles;
integralError = integralError + errorAngles;
errorVel = (errorAngles - prevError) / timeStep;
torques = K(:,1:4).*errorAngles ...
        + K(:,5:8).*errorVel ...
        + K(:,9:12).*integralError;
```

**Verified source issues:**

- The documented integral gains are applied to the derivative term, and the documented derivative gains are applied to the integral term.
- `angles` is not updated after the joint coordinates change, so the error is not computed from the current simulated state.
- The integral accumulator is not multiplied by `timeStep`.
- `torques' * inputTorques` forms a 4-by-4 outer product rather than a four-element actuation vector.
- The prismatic position update uses `angularAccel(3)` instead of `angularAccel(4)`.
- The simulation stops when any coordinate exceeds its target, leaving trailing zeros in preallocated arrays.

### 5.3 Preserved PID results

![PID joint positions](../assets/images/pid-position-response.png)

![PID joint velocities](../assets/images/pid-velocity-response.png)

![PID arm configuration](../assets/images/pid-arm-configuration.jpg)

The preserved numerical claim is

$$\theta_1=90.0025^\circ,\quad \theta_2=89.998^\circ,\quad \theta_3=90.0495^\circ,\quad d_4=40.3407\;\mathrm{mm}.$$

### 5.4 PID interpretation

The rising portions of the preserved curves are consistent with motion toward the requested coordinates. The subsequent drop to zero is not a physical return to the origin; it is explained by the early loop termination and untouched trailing elements in the preallocated arrays. The source narrative reports both 0.4 s and 10 s, while the plotted transition occurs at approximately 10 s. The terminal values are therefore retained as historical claims, not as independently reproduced controller performance.

## 6. Part II - learning-based sampling theory

The referenced method learns a conditional sampling distribution for sampling-based motion planning. A Conditional Variational Autoencoder models

$$p(x\mid y)=\int p(x\mid z,y)p(z\mid y)\,dz,$$

where $x$ is a useful planning state, $y$ describes the problem, and $z$ is a latent variable. Training maximizes an evidence lower bound (ELBO), conventionally written as

$$\mathcal{L}_{ELBO}=\mathbb{E}_{q_\phi(z\mid x,y)}[\log p_\theta(x\mid z,y)]-D_{KL}(q_\phi(z\mid x,y)\|p(z\mid y)).$$

A hybrid sampler combines learned and uniform samples so that learned concentration does not completely remove broad state-space coverage.

**Theory-to-code boundary:** no encoder, decoder, latent network, ELBO optimization, occupancy-grid condition, or uniform sampling-based motion planner is implemented in this repository. The equations above explain the research inspiration, not the executed Python algorithm.

## 7. Part II - repository implementation

The full implementation is in [`src/python/part2_control.py`](../src/python/part2_control.py). It contains four main classes.

| Class | Responsibility |
|---|---|
| `HedgeTrimmingRobot` | Joint limits and forward kinematics |
| `Environment` | Three spherical obstacles and collision/proximity queries |
| `NormalizingFlowMPC` | Synthetic trajectory generation, GMM fitting, cost evaluation, and control selection |
| `MotionPlanningSimulation` | Three scenarios, state integration, figures, GIFs, and summary output |

The class name `NormalizingFlowMPC` is historical. The learned object is `sklearn.mixture.GaussianMixture`.

### 7.1 Synthetic training and GMM fitting

For each scenario, the program attempts to generate 200 trajectories using straight, via-point, random, and obstacle-avoidance heuristics. Feasible trajectories are flattened and used to fit a three-component full-covariance GMM.

```python
training_data = np.array(training_trajectories)
self.learned_distribution = GaussianMixture(
    n_components=min(self.n_components, len(training_trajectories)//10),
    covariance_type='full',
    random_state=42
)
self.learned_distribution.fit(training_data)
```

This learns a distribution over complete discretized trajectories for one start-goal pair. It is not an environment-conditioned model that generalizes across planning problems.

### 7.2 MPC-style objective and action selection

For candidate trajectory $q_0,\ldots,q_{N-1}$, the implemented objective can be summarized as

$$
J=\sum_t \left[w_t e_{q,t}^TQ_{pos}e_{q,t}+50w_t\|p(q_t)-p(q_g)\|^2+\Delta q_t^TR\Delta q_t+J_{obs,t}+0.1\left\|\frac{\Delta q_t}{\Delta t}\right\|^2\right]+100\|q_{N-1}-q_g\|^2.
$$

Candidates violating a joint limit or reported collision receive infinite cost. The controller applies the first difference from the lowest-cost trajectory:

```python
control = (best_trajectory[1] - best_trajectory[0]) / self.dt
control *= 2.0
return np.clip(control, -1.0, 1.0)
```

The simulated state is then advanced by $q_{k+1}=q_k+\dot q_k\Delta t$. There is no rigid-body state transition, acceleration state, torque model, or numerical optimizer inside this control loop. `scipy.optimize.minimize` is used only by an approximate inverse-kinematics helper.

### 7.3 Collision-checking implementation

The tracked collision function checks each reported joint position against each sphere with a 0.02 m margin:

```python
for pos in positions:
    for obs in self.obstacles:
        if np.linalg.norm(pos - obs['center']) <= obs['radius'] + 0.02:
            return True
```

The source checks discrete joint positions. It does not compute the distance from an obstacle to the link segment between consecutive joints.

### 7.4 Preserved planning results

#### Point-to-point

![Point-to-point task space](../assets/images/planning-task-space-point-to-point.png)

![Point-to-point configuration space](../assets/images/planning-configuration-space-point-to-point.png)

#### Complex maneuvering

![Complex task space](../assets/images/planning-task-space-complex-maneuvering.png)

![Complex configuration space](../assets/images/planning-configuration-space-complex-maneuvering.png)

### 7.5 Planning-result interpretation

The point-to-point figures show a smooth end-effector curve and convergence toward the displayed joint targets, which is qualitatively consistent with the goal-tracking and smoothness terms in the implemented cost. The complex trajectory contains non-monotonic corrections, especially in the prismatic coordinate, which is consistent with repeated candidate selection in the presence of obstacle penalties. These are preserved historical outputs, not newly reproduced benchmarks. Source screenshots associated with the figures differ from the tracked scenario definitions, so the exact run configuration remains unresolved.

### 7.6 Reconstructed collision diagnostic - result

A read-only diagnostic sampled 101 configurations along each direct start-goal interpolation and compared the source's endpoint checks with zero-radius link-segment checks using the same 0.02 m margin.

| Scenario | Endpoint-collision samples | Segment-collision samples | Missed by source check |
|---|---:|---:|---:|
| Point-to-Point | 0 | 0 | 0 |
| Obstacle Avoidance | 0 | 39 | 39 |
| Complex Maneuvering | 22 | 31 | 9 |

### 7.7 Reconstructed collision diagnostic - interpretation

The diagnostic shows that point-only collision checking can miss link-obstacle intersections, most clearly in the obstacle-avoidance scenario. These values are reconstructed diagnostics rather than preserved project results. They establish a limitation of the collision test, but they do not determine the collision status of every sampled candidate or every historical plotted trajectory.

## 8. Results inventory

| Artifact | Status | What can be claimed |
|---|---|---|
| Open-loop position and velocity figures | Preserved result | A historical script produced the displayed curves |
| Open-loop arm configuration | Preserved result | A historical script produced the displayed stick figure |
| PID position and velocity figures | Preserved result | Historical controller output with known trailing-zero behavior |
| PID terminal values | Reported numerical result | Values were supplied, but not independently reproduced |
| Planning task-space figures | Preserved result | Historical trajectories were plotted with three spherical obstacles |
| Planning configuration-space figures | Preserved result | Historical coordinates approached the displayed targets |
| Collision comparison table | Reconstructed diagnostic | Point-only checks can miss link-obstacle intersections |
| CVAE performance claims | Research theory only | Not a result of the tracked implementation |

No raw time series, random seeds for the complete historical runs, dependency lockfile, hardware timing log, or classical-planner baseline was supplied.

## 9. Reproduction guide

### MATLAB

Open MATLAB, change to `src/matlab/`, and run:

```matlab
run('part1_not_pid.m')
run('part1_pid.m')
```

The scripts open figures but do not save result files. Because the historical MATLAB version is unknown and the verified source issues affect numerical behavior, a new run should be labeled as a reproduction attempt rather than assumed to match the preserved figures.

### Python

From the repository root:

```bash
python -m venv .venv
python -m pip install -r requirements.txt
python src/python/part2_control.py
```

Activate the virtual environment before the final two commands. The output directory is created relative to the current working directory. Historical package versions are unresolved, so `requirements.txt` lists direct dependencies without claiming an exact lock.

## 10. Engineering limitations and recommended next work

### Verified limitations

1. Normalize all MATLAB parameters to one physical unit system.
2. Re-derive or validate $M(q)$, $C(q,\dot q)$, and $G(q)$ before interpreting dynamic performance.
3. Update state vectors every integration step and correct the PID gain mapping.
4. Replace the PID outer-product actuation with an explicit four-element vector.
5. Correct the prismatic acceleration index and integration rule.
6. Add segment or capsule collision checking for every robot link.
7. Rename the Python planner to reflect its GMM implementation, or implement the stated CVAE method as a separate module.
8. Add deterministic tests for forward kinematics, limits, collision queries, trajectory costs, and output creation.
9. Preserve raw result arrays and a machine-readable run manifest for every regenerated figure.
10. Compare learned sampling against an explicit uniform baseline before making performance claims.

These changes should be handled as a separate engineering revision. Correcting the source will change numerical behavior and would invalidate direct comparison with the preserved historical figures unless both versions are retained and labeled.

## 11. Conclusions

The repository demonstrates two useful robotics workflows: a compact MATLAB study of simplified joint dynamics and PID control, and a Python study of trajectory-distribution learning and MPC-style action selection. Its strongest educational value comes from connecting equations, source structure, and visualization.

The central documentation conclusion is equally important: the referenced CVAE method, the historical figures, and the tracked implementation are related but not identical. Treating them as separate evidence layers preserves the technical character of the project without overstating reproducibility or algorithmic capability.

### Referenced research method

Ichter, B., Harrison, J., and Pavone, M., *Learning Sampling Distributions for Robot Motion Planning*. The paper motivates conditional learned sampling for sampling-based motion planning; the tracked repository uses a simplified GMM trajectory model instead.